## Code to plot neuron morphologies and synapses from FlyWire
fafbseg probably only works on MacOS
Before the first time using it, you need to create API token, see here: https://fafbseg-py.readthedocs.io/en/latest/source/tutorials/flywire_setup.html

In [ ]:
import fafbseg.flywire as fw
import navis
import pandas as pd
from tqdm import tqdm
import plotly.io as pio
import numpy as np
import matplotlib.pyplot as plt
import pathlib
pio.renderers.default = 'iframe'

pd.options.mode.copy_on_write = True # prevent SettingWithCopy warning
%config InlineBackend.figure_format = "retina" # looks better on Mac

savepath_data = pathlib.Path.cwd() / 'output_data' 
savepath_data.mkdir(parents=True, exist_ok=True)
savepath_fig = pathlib.Path.cwd() / 'output_fig' 
savepath_fig.mkdir(parents=True, exist_ok=True)

# For illustrator figures, so that text appears in text boxes
plt.rc('pdf', fonttype = 42)

# fw.set_default_dataset('flat_783') # using the published snapshot of FlyWire with synapse predictions
fw.set_default_dataset("public")

## Dendrite plotting
Uses navis.split_axon_dendrite to separate neuron into dendrite and axon, and sometimes also linker region. Used for plotting to determine axon root and keep only the dendrite and its synapses in the final plot. Coloring of synapses is done here based on 'type' in the connectors dataframe. Here I used the predicted transmitter of each input neuron to color the synapses, and I override the default values for synapses on the linker/axon (which I call 'linker_cn' and 'axon_cn') to easily exclude these later.

'dendrite_inputs' is the more extensive info dataframe about the dendritic synapses before reducing it down to the one needed for plotting. Useful if you want to find info about which neuron made which synapse, but not used for the plotting.

In [ ]:
def neuron_parts_and_inputs(root_id):
    n = fw.get_mesh_neuron(root_id)
    syn_df = fw.synapses.get_synapses(root_id).rename(columns={'post_x':'x', 'post_y':'y','post_z':'z'})
    syn_df["type"] = np.where(syn_df["pre"] == root_id, "pre", "post")
    n.connectors = syn_df
    n = navis.split_axon_dendrite(n)
    
    for neuron_part in n:
        if neuron_part.compartment == 'dendrite': # only label synapses on the dendrite
            dendrite = neuron_part
            dendrite.connectors = dendrite.connectors.query("pre != @root_id") # only presynapses to the neuron
            
            transmitter_lookup = {}
            for input_cell in tqdm(dendrite.connectors.pre.unique()):
                transmitter, probability = fw.synapses.get_transmitter_predictions(input_cell, single_pred=True, materialization=783)[input_cell]
                transmitter_lookup[input_cell] = {'transmitter': transmitter, 'probability': probability}
            
            transmitter_lookup_df = pd.DataFrame(transmitter_lookup).T.rename_axis('input_cell').reset_index() 
            synapse_plotting = dendrite.connectors.copy()
            synapse_plotting = pd.merge(left=synapse_plotting, right=transmitter_lookup_df, how='left', left_on='pre', right_on='input_cell')
            dendrite_inputs = synapse_plotting
            dendrite.connectors = synapse_plotting[['x', 'y', 'z', 'transmitter']].rename(columns={'transmitter':'type'})
            
        elif neuron_part.compartment == 'linker':
            linker = neuron_part
            linker.connectors = linker.connectors.query("pre != @root_id") # only presynapses to the neuron
            
            transmitter_lookup = {}
            for input_cell in tqdm(linker.connectors.pre.unique()):
                transmitter, probability = fw.synapses.get_transmitter_predictions(input_cell, single_pred=True, materialization=783)[input_cell]
                transmitter_lookup[input_cell] = {'transmitter': transmitter, 'probability': probability}
            
            transmitter_lookup_df = pd.DataFrame(transmitter_lookup).T.rename_axis('input_cell').reset_index()
            
            synapse_plotting = linker.connectors.copy()
            synapse_plotting = pd.merge(left=synapse_plotting, right=transmitter_lookup_df, how='left', left_on='pre', right_on='input_cell')

            try:
                linker.connectors = synapse_plotting[['x', 'y', 'z', 'transmitter']].rename(columns={'transmitter':'type'})
                # linker.connectors.type = 'linker_cn'
            except:
                print('Linker does not have synapses')
            
        elif neuron_part.compartment == 'axon':
            axon = neuron_part
            axon.connectors = axon.connectors.query("pre != @root_id") # only presynapses to the neuron
            
            transmitter_lookup = {}
            for input_cell in tqdm(axon.connectors.pre.unique()):
                transmitter, probability = fw.synapses.get_transmitter_predictions(input_cell, single_pred=True, materialization=783)[input_cell]
                transmitter_lookup[input_cell] = {'transmitter': transmitter, 'probability': probability}
            
            transmitter_lookup_df = pd.DataFrame(transmitter_lookup).T.rename_axis('input_cell').reset_index()
            
            synapse_plotting = axon.connectors.copy()
            synapse_plotting = pd.merge(left=synapse_plotting, right=transmitter_lookup_df, how='left', left_on='pre', right_on='input_cell')
    
            axon.connectors = synapse_plotting[['x', 'y', 'z', 'transmitter']].rename(columns={'transmitter':'type'})
            # axon.connectors.type = 'axon_cn'
    return (dendrite, axon, linker, dendrite_inputs) 

def get_neuron_and_synapses(root_id): # for whole neurons
    n = fw.get_mesh_neuron(root_id)
    syn_df = fw.synapses.get_synapses(root_id).rename(columns={'post_x':'x', 'post_y':'y','post_z':'z'})
    syn_df["type"] = np.where(syn_df["pre"] == root_id, "pre", "post")
    postsynapses = syn_df.query("type == 'post'")
    n.connectors = postsynapses # only synapses with root neuron as postsynapse
    
    transmitter_lookup = {}
    for input_cell in tqdm(n.connectors.pre.unique()):
        transmitter, probability = fw.synapses.get_transmitter_predictions(input_cell, single_pred=True, materialization=783)[input_cell]
        transmitter_lookup[input_cell] = {'transmitter': transmitter, 'probability': probability}
    
    transmitter_lookup_df = pd.DataFrame(transmitter_lookup).T.rename_axis('input_cell').reset_index()
   
    synapse_plotting = postsynapses.copy()
    synapse_plotting = pd.merge(left=synapse_plotting, right=transmitter_lookup_df, how='left', left_on='pre', right_on='input_cell')
    
    connectors = synapse_plotting[['x', 'y', 'z', 'transmitter']].rename(columns={'transmitter':'type'})
    n.connectors = connectors

    return n, synapse_plotting

In [ ]:
# T4c
T4c_dendrite, T4c_axon, T4c_linker, T4c_dendrite_inputs = neuron_parts_and_inputs(720575940612221669)

In [ ]:
# T5c
T5c_dendrite, T5c_axon, T5c_linker, T5c_dendrite_inputs = neuron_parts_and_inputs(720575940627480842)

## T4c plotting

In [ ]:
# Interactive plot (no vector graphics export possible)
navis.plot3d([T4c_dendrite, T4c_linker], 
             color=['grey','red'], 
             fig_autosize=True, 
             connectors=True, 
             alpha=0.2, 
             cn_alpha=0.8, 
             cn_size=10, 
             cn_colors={'glutamate': '#75bc64', 'acetylcholine':'#52737c', 'gaba':'#52c3c3', 'linker_cn':'red'})

In [ ]:
# Matplotlib plot 2d/3d projection (vector graphics export)
fig, ax = navis.plot2d([T4c_dendrite, T4c_linker], 
                       method='3d', 
                       mesh_shade=True, 
                       color=['grey','red'], 
                       connectors=True, 
                       alpha=0.1, 
                       cn_alpha=0.8, 
                       cn_size=30, 
                       cn_layout = {'glutamate': {'color':'#75bc64'}, 'acetylcholine':{'color':'#52737c'}, 'gaba':{'color':'#52c3c3'}, 'linker_cn':{'color':'red'}})
ax.view_init(elev=-40, azim=40, roll=0)
ax.grid(visible=False)

plt.savefig(savepath_fig / 'T4c_720575940612221669.pdf')

## T5c plotting

In [ ]:
# Interactive plot (no vector graphics export possible)
navis.plot3d([T5c_dendrite, T5c_linker], 
             color=['grey','red'], 
             fig_autosize=True, 
             connectors=True, 
             alpha=0.2, 
             cn_alpha=0.8, 
             cn_size=10, 
             cn_colors={'glutamate': '#75bc64', 'acetylcholine':'#52737c', 'gaba':'#52c3c3', 'linker_cn':'red'})

In [ ]:
# Matplotlib plot 2d/3d projection (vector graphics export)
fig, ax = navis.plot2d([T5c_dendrite, T5c_linker], 
                       method='3d', 
                       mesh_shade=True, 
                       color=['grey','red'], 
                       connectors=True, 
                       alpha=0.1, 
                       cn_alpha=0.8, 
                       cn_size=30, 
                       cn_layout = {'glutamate': {'color':'#75bc64'}, 'acetylcholine':{'color':'#52737c'}, 'gaba':{'color':'#52c3c3'}, 'linker_cn':{'color':'red'}})
ax.view_init(elev=90, azim=-90, roll=0)
ax.grid(visible=False)

plt.savefig(savepath_fig / 'T5c_720575940627480842.pdf')

# Exports for computational model of T4

In [ ]:
T4c_full, T4c_full_synapse_table = get_neuron_and_synapses(720575940612221669)

T4c_dendrite.connectors.to_csv(savepath_data / 'T4_connectors_dendrite.csv')
T4c_axon.connectors.to_csv(savepath_data / 'T4_connectors_axon.csv')
T4c_linker.connectors.to_csv(savepath_data / 'T4_connectors_linker.csv')
T4c_full.connectors.to_csv(savepath_data / 'T4_connectors_full.csv')

In [ ]:
# Make T4 skeleton for SWC
T4c_skeleton = navis.conversion.mesh2skeleton(T4c_full, connectors=True, heal=True)
T4c_dendrite_skeleton = navis.conversion.mesh2skeleton(T4c_dendrite, connectors=True, heal=True)

inputs_table_dendrite = fw.search_annotations(T4c_dendrite_inputs.input_cell.unique(), materialization=783)
inputs_table = fw.search_annotations(T4c_full_synapse_table.input_cell.unique(), materialization=783)

ann = fw.search_annotations(T4c_full_synapse_table.input_cell.unique(), materialization=783)
cell_type = pd.DataFrame({"root_id": T4c_full_synapse_table.input_cell.unique()})
cell_type = cell_type.merge(
    ann[["root_id", "cell_type"]],
    on="root_id",
    how="left"
)

ann_dendrite = fw.search_annotations(T4c_dendrite_inputs.input_cell.unique(), materialization=783)
cell_type_dendrite = pd.DataFrame({"root_id": T4c_dendrite_inputs.input_cell.unique()})
cell_type_dendrite = cell_type_dendrite.merge(
    ann_dendrite[["root_id", "cell_type"]],
    on="root_id",
    how="left"
)

connectors_more_info = pd.concat([T4c_skeleton.connectors, T4c_full_synapse_table['input_cell']], axis=1)
connectors_more_info['cell_type'] = connectors_more_info.apply(lambda row: cell_type.query("root_id == @row.input_cell").cell_type.values[0], axis=1)

connectors_more_info_dendrite = pd.concat([T4c_dendrite_skeleton.connectors, T4c_dendrite_inputs['input_cell']], axis=1)
connectors_more_info_dendrite['cell_type'] = connectors_more_info_dendrite.apply(lambda row: cell_type_dendrite.query("root_id == @row.input_cell").cell_type.values[0], axis=1)

T4c_skeleton.connectors = connectors_more_info
T4c_dendrite_skeleton.connectors = connectors_more_info_dendrite

node_map = navis.write_swc(x=T4c_skeleton, 
                filepath=savepath_data / 'T4_skeleton_full.swc', 
                export_connectors=True, 
                write_meta=True, 
                labels=True, 
                return_node_map=True,)

node_map_dendrite = navis.write_swc(x=T4c_dendrite_skeleton, 
                filepath=savepath_data / 'T4_skeleton_dendrite.swc', 
                export_connectors=True, 
                write_meta=True, 
                labels=True, 
                return_node_map=True,)

connectors_more_info['swc_node'] = connectors_more_info.node_id.map(node_map)
connectors_more_info.to_csv(savepath_data / 'T4_synapse_info.csv')

connectors_more_info_dendrite['swc_node'] = connectors_more_info_dendrite.node_id.map(node_map_dendrite)
connectors_more_info_dendrite.to_csv(savepath_data / 'T4_synapse_info_dendrite.csv')

navis.write_mesh(T4c_dendrite, savepath_data / 'T4_dendrite_mesh.obj')
navis.write_mesh(T4c_full, savepath_data / 'T4_full_mesh.obj')